# 第 2 章：提示工程与结构化输出

Pydantic ≈ Zod —— 让 LLM 输出结构化数据而非纯文本。

## 2.1 Pydantic ≈ Zod

前端开发者已经熟悉 Zod schema 验证：

```typescript
const PersonSchema = z.object({
  name: z.string(),
  age: z.number(),
});
```

Python 中等价的是 Pydantic BaseModel：

```python
class Person(BaseModel):
    name: str
    age: int
```


In [ ]:
from pydantic import BaseModel


class Person(BaseModel):
    name: str
    age: int
    occupation: str


class Company(BaseModel):
    name: str
    founded: int
    industry: str


# Pydantic 自动把 JSON → Python 对象（像 Zod.parse()）
p = Person.model_validate_json('{"name": "张三", "age": 30, "occupation": "工程师"}')
print(f"{p.name}, {p.age}岁, {p.occupation}")

## 2.2 结构化抽取：手写 JSON parse 版

核心思路：让 LLM 返回 JSON 字符串 → 用 Pydantic 解析。

这是看清底层的第一步。后面会看到 `with_structured_output()` 自动完成这个过程。

In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel

from structured_extractor import extract_with_schema

llm = FakeListChatModel(
    responses=['{"name": "张三", "age": 30, "occupation": "工程师"}'],
)

result = extract_with_schema(llm, "张三，30岁，是一名软件工程师。", Person)
print(f"抽取结果: {result}")

## 2.3 ChatPromptTemplate

类比前端：
- `ChatPromptTemplate` ≈ JSX 模板 + props
- `MessagesPlaceholder` ≈ React slot

把硬编码的字符串变成可复用的模板。

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

# 创建模板（类似 React component）
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "你是{role}，只回答{domain}问题。"),
        ("human", "{question}"),
    ]
)


# 注入变量（类似 JSX props）
messages = prompt.invoke({"role": "数学老师", "domain": "数学", "question": "1+1=?"}).to_messages()
print(f"System: {messages[0].content}")
print(f"Human: {messages[1].content}")

## 2.4 Chain of Thought（思维链）

类比前端：分步调试——让 LLM "Let's think step by step"，
就像把一个大函数拆成多个小步骤 debug。

In [ ]:
from langchain_community.chat_models.fake import FakeListChatModel

# CoT: 在 prompt 中要求 LLM 分步推理
llm = FakeListChatModel(
    responses=["1. 理解问题: 求 17 × 23\n2. 计算: 17 × 23 = 391\n3. 答案: 391"],
)

result = llm.invoke("请分步计算 17 × 23")
print(result.content)